**Gradient Boost Regresser**


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error


"""
    Gradient Boosting Regressor from scratch using MSE loss.
"""




class GradientBoostReg:
    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.trees = []
        self.f0 = None
        self.losses = []

    def fit(self, X, y):
        # Initial prediction
        self.f0 = np.mean(y)
        y_pred = np.full(y.shape, self.f0, dtype=float)

        for i in range(self.n_estimators):
            residuals = y - y_pred

            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)

            y_pred += self.learning_rate * tree.predict(X)

            mse = mean_squared_error(y, y_pred)
            self.losses.append(mse)

            if (i + 1) % 10 == 0 or i == 0:
                print(f"Iter {i+1}/{self.n_estimators}, MSE: {mse:.4f}")

    def predict(self, X):
        y_pred = np.full(X.shape[0], self.f0, dtype=float)
        for tree in self.trees:
            y_pred += self.learning_rate * tree.predict(X)
        return y_pred


In [ ]:


# Simple regression dataset
X_reg = np.array([[1], [2], [3], [4], [5], [6]])
y_reg = np.array([1.2, 1.9, 3.0, 3.8, 5.1, 5.9])

# Initialize model
gbr = GradientBoostReg(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=2
)

# Train
gbr.fit(X_reg, y_reg)

# Predict
pred_reg = gbr.predict(X_reg)

print("Actual values:   ", y_reg)
print("Predicted values:", np.round(pred_reg, 2))



Iter 1/50, MSE: 2.2465
Iter 10/50, MSE: 0.3786
Iter 20/50, MSE: 0.0526
Iter 30/50, MSE: 0.0073
Iter 40/50, MSE: 0.0010
Iter 50/50, MSE: 0.0001
Actual values:    [1.2 1.9 3.  3.8 5.1 5.9]
Predicted values: [1.22 1.91 3.   3.8  5.09 5.88]


**Gradient Boost Classifier**

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score

"""
Gradient Boosting Classifier from scratch using log-loss.
"""



class GradientBoostingClassifierScratch:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        # Initial log-odds (safe)
        pos_ratio = np.clip(np.mean(y), 1e-5, 1 - 1e-5)
        self.F0 = np.log(pos_ratio / (1 - pos_ratio))

        Fm = np.full(y.shape, self.F0, dtype=float)

        for i in range(self.n_estimators):
            p = self._sigmoid(Fm)

            # Negative gradient of log-loss
            residuals = y - p

            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)

            Fm += self.learning_rate * tree.predict(X)

            if (i + 1) % 10 == 0 or i == 0:
                acc = accuracy_score(y, self.predict(X))
                print(f"Iter {i+1}/{self.n_estimators}, Train Acc: {acc:.4f}")

    def predict_proba(self, X):
        Fm = np.full(X.shape[0], self.F0, dtype=float)
        for tree in self.trees:
            Fm += self.learning_rate * tree.predict(X)
        return self._sigmoid(Fm)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


In [ ]:
# Binary classification dataset
X_clf = np.array([
    [1], [2], [3], [4], [5], [6], [7], [8]
])

y_clf = np.array([0, 0, 0, 0, 1, 1, 1, 1])



# Initialize model
gbc = GradientBoostingClassifierScratch(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=2
)

# Train
gbc.fit(X_clf, y_clf)

# Predict probabilities
proba = gbc.predict_proba(X_clf)

# Predict classes
pred_clf = gbc.predict(X_clf)

print("Actual labels:     ", y_clf)
print("Predicted labels:  ", pred_clf)
print("Predicted probas:  ", np.round(proba, 3))


Iter 1/50, Train Acc: 1.0000
Iter 10/50, Train Acc: 1.0000
Iter 20/50, Train Acc: 1.0000
Iter 30/50, Train Acc: 1.0000
Iter 40/50, Train Acc: 1.0000
Iter 50/50, Train Acc: 1.0000
Actual labels:      [0 0 0 0 1 1 1 1]
Predicted labels:   [0 0 0 0 1 1 1 1]
Predicted probas:   [0.181 0.181 0.181 0.181 0.819 0.819 0.819 0.819]
